In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("weather-explore").getOrCreate()

raw = spark.read.json("/home/jovyan/work/data/raw/")
raw.printSchema()
print(raw.count())

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- hourly_units: struct (nullable = true)
 |    |-- precipitation: string (nullable = true)
 |    |-- temperature_2m: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (nullable = true)
 |-- date: date (nullable = true)

10


In [6]:
from pyspark.sql import functions as F

pattern = r"([a-z]+)_(backfill|daily)\.json$"

with_city = (
    raw
    .withColumn("file_path", F.col("_metadata.file_path"))
    .withColumn("city", F.regexp_extract("file_path", pattern, 1))
    .withColumn("mode", F.regexp_extract("file_path", pattern, 2))
)

with_city.select("city", "mode", "date").show()

+----------+--------+----------+
|      city|    mode|      date|
+----------+--------+----------+
|     rabat|backfill|2026-10-27|
|casablanca|backfill|2026-10-27|
|   tangier|backfill|2026-10-27|
| marrakech|backfill|2026-10-27|
|       fes|backfill|2026-10-27|
|   tangier|   daily|2026-10-27|
|       fes|   daily|2026-10-27|
| marrakech|   daily|2026-10-27|
|     rabat|   daily|2026-10-27|
|casablanca|   daily|2026-10-27|
+----------+--------+----------+



In [8]:
hourly = (
    with_city
    .select(
        "city",
        "mode",
        "date",                                              # NEW
        F.col("hourly.time").alias("time"),
        F.col("hourly.temperature_2m").alias("temp"),
        F.col("hourly.precipitation").alias("precip"),
    )
    .withColumn("h", F.explode(F.arrays_zip("time", "temp", "precip")))
    .select(
        "city",
        "mode",
        F.col("date").alias("extracted_on"),                 # NEW
        F.to_timestamp("h.time", "yyyy-MM-dd'T'HH:mm").alias("observed_at"),
        F.col("h.temp").alias("temperature_c"),
        F.col("h.precip").alias("precipitation_mm"),
    )
)

hourly.show(5)
print(hourly.count())

+-----+--------+------------+-------------------+-------------+----------------+
| city|    mode|extracted_on|        observed_at|temperature_c|precipitation_mm|
+-----+--------+------------+-------------------+-------------+----------------+
|rabat|backfill|  2026-10-27|2021-01-01 00:00:00|          9.8|             0.0|
|rabat|backfill|  2026-10-27|2021-01-01 01:00:00|         10.1|             0.0|
|rabat|backfill|  2026-10-27|2021-01-01 02:00:00|          9.2|             0.0|
|rabat|backfill|  2026-10-27|2021-01-01 03:00:00|          9.0|             0.0|
|rabat|backfill|  2026-10-27|2021-01-01 04:00:00|          9.3|             0.0|
+-----+--------+------------+-------------------+-------------+----------------+
only showing top 5 rows
255120


In [10]:
from pyspark.sql import Window

priority = F.when(F.col("mode") == "backfill", 0).otherwise(1)

w = Window.partitionBy("city", "observed_at").orderBy(
    F.col("extracted_on").desc(), priority
)

deduped = (
    hourly
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

deduped.cache()
print(deduped.count())

check = (F.col("city") == "casablanca") & (F.col("observed_at") == "2026-10-20 12:00:00")
hourly.filter(check).show()
deduped.filter(check).show()

255000
+----------+--------+------------+-------------------+-------------+----------------+
|      city|    mode|extracted_on|        observed_at|temperature_c|precipitation_mm|
+----------+--------+------------+-------------------+-------------+----------------+
|casablanca|backfill|  2026-10-27|2026-10-20 12:00:00|         23.4|             0.0|
|casablanca|   daily|  2026-10-27|2026-10-20 12:00:00|         23.8|             0.0|
+----------+--------+------------+-------------------+-------------+----------------+

+----------+--------+------------+-------------------+-------------+----------------+
|      city|    mode|extracted_on|        observed_at|temperature_c|precipitation_mm|
+----------+--------+------------+-------------------+-------------+----------------+
|casablanca|backfill|  2026-10-27|2026-10-20 12:00:00|         23.4|             0.0|
+----------+--------+------------+-------------------+-------------+----------------+



In [11]:
# Completeness: count the nulls in every column
deduped.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in deduped.columns]
).show()

# Validity and coverage: one summary row per city
deduped.groupBy("city").agg(
    F.count("*").alias("hours"),
    F.min("observed_at").alias("first_hour"),
    F.max("observed_at").alias("last_hour"),
    F.min("temperature_c").alias("min_temp"),
    F.max("temperature_c").alias("max_temp"),
    F.min("precipitation_mm").alias("min_precip"),
).orderBy("city").show(truncate=False)

# Uniqueness: total rows vs distinct city+hour pairs
print(deduped.count(), deduped.select("city", "observed_at").distinct().count())

+----+----+------------+-----------+-------------+----------------+
|city|mode|extracted_on|observed_at|temperature_c|precipitation_mm|
+----+----+------------+-----------+-------------+----------------+
|   0|   0|           0|          0|            0|               0|
+----+----+------------+-----------+-------------+----------------+

+----------+-----+-------------------+-------------------+--------+--------+----------+
|city      |hours|first_hour         |last_hour          |min_temp|max_temp|min_precip|
+----------+-----+-------------------+-------------------+--------+--------+----------+
|casablanca|51000|2021-01-01 00:00:00|2026-10-26 23:00:00|3.6     |40.8    |0.0       |
|fes       |51000|2021-01-01 00:00:00|2026-10-26 23:00:00|-2.3    |46.4    |0.0       |
|marrakech |51000|2021-01-01 00:00:00|2026-10-26 23:00:00|0.2     |48.0    |0.0       |
|rabat     |51000|2021-01-01 00:00:00|2026-10-26 23:00:00|5.1     |42.8    |0.0       |
|tangier   |51000|2021-01-01 00:00:00|2026-

In [12]:
daily = (
    deduped
    .withColumn("day", F.to_date("observed_at"))
    .groupBy("city", "day")
    .agg(
        F.round(F.min("temperature_c"), 1).alias("temp_min_c"),
        F.round(F.max("temperature_c"), 1).alias("temp_max_c"),
        F.round(F.avg("temperature_c"), 1).alias("temp_avg_c"),
        F.round(F.sum("precipitation_mm"), 1).alias("precip_total_mm"),
        F.count("*").alias("hours_recorded"),
    )
)

In [13]:
w7 = Window.partitionBy("city").orderBy("day").rowsBetween(-6, 0)

daily = (
    daily
    .withColumn("temp_avg_7d_c", F.round(F.avg("temp_avg_c").over(w7), 1))
    .withColumn("is_rainy_day", F.col("precip_total_mm") >= 1.0)
)

print(daily.count())
daily.filter(F.col("city") == "marrakech").orderBy(F.col("day").desc()).show(10)

10625
+---------+----------+----------+----------+----------+---------------+--------------+-------------+------------+
|     city|       day|temp_min_c|temp_max_c|temp_avg_c|precip_total_mm|hours_recorded|temp_avg_7d_c|is_rainy_day|
+---------+----------+----------+----------+----------+---------------+--------------+-------------+------------+
|marrakech|2026-10-26|      15.0|      32.3|      23.2|            0.0|            24|         21.7|       false|
|marrakech|2026-10-25|      16.2|      30.5|      22.9|            0.0|            24|         21.0|       false|
|marrakech|2026-10-24|      16.3|      29.2|      22.3|            0.0|            24|         20.5|       false|
|marrakech|2026-10-23|      17.5|      28.4|      22.6|            0.0|            24|         20.7|       false|
|marrakech|2026-10-22|      17.0|      28.0|      21.9|            0.0|            24|         21.0|       false|
|marrakech|2026-10-21|      13.1|      27.2|      19.8|            0.0|           

In [15]:
base = "/home/jovyan/work/data/processed"

deduped.write.mode("overwrite").partitionBy("city").parquet(f"{base}/weather_hourly")
daily.write.mode("overwrite").parquet(f"{base}/weather_daily")

print(spark.read.parquet(f"{base}/weather_hourly").count())
print(spark.read.parquet(f"{base}/weather_daily").count())

255000
10625
